# mega golf probe — train from the relabel bundle (any GPU; teacher never loads)

Input = `relabel_bundle.zip` from `sae_relabel_pipeline.ipynb` (9B-teacher firings, 2,045 fresh captions,
874 gated, corpus 32k BPE). This notebook trains the student and answers each question with ONE variable
changed at a time — the ladder is the confound control:

| rung | change | isolates |
|---|---|---|
| A | old nano arch, teacher-token hashes, new data+dictionary | data/dictionary effect |
| B | = A but **new 32k tokenizer** (targets re-aligned by char overlap) | tokenizer effect |
| C | = B but **factorized embeddings** (d=64 tables -> proj 256) | size effect |
| D | = C + **entropy golf**: log-prior bias head + soft distillation on teacher act distributions | the new idea |
| E | = D but **smear gate replaces the bigram table** (13 params vs ~1M; parameter-golf PR #1667) | compute-vs-memorize |

Then: surprise lens (per-token concept entropy + adjacent-token KL -> decision boundaries, VALIDATED against
the teacher's own label-set changes), honest metrics per domain, LLM judge comparing rung A vs rung D output,
optional teacher spot-check. Old-notebook numbers (0.83 AUC etc.) are context only — different teacher,
corpus and dictionary; never compare across notebooks, only across rungs.

In [1]:
# 1 · install (no transformer-lens, no sae-lens — the teacher is already harvested)
!pip -q install sentencepiece scikit-learn

In [2]:
# 2 · config
BUNDLE = "/content/relabel_bundle"                 # unzip target (auto-unpacks relabel_bundle.zip if present)
CDIM, NHEAD, NLAYER, MAXLEN, BIG_H = 256, 4, 2, 112, 16384
FDIM = 64                                          # factorized table dim (rungs C, D)
EPOCHS, BS, LR, SEED, TRAIN_FRAC = 15, 96, 3e-3, 0, 0.7
TEMP, SOFT_W = 2.0, 0.3                            # soft-distill temperature + aux weight (rung D)
JUDGE_MODEL, N_JUDGE = "google/gemini-2.5-flash-lite", 8
LOAD_TEACHER = False                               # True + A100 -> optional teacher agreement spot-check (last cell)
import os
OPENROUTER_KEY = os.environ.get("OPENROUTER_KEY", "")
print("config ready")

config ready


In [3]:
# 3 · load the bundle: spans, per-token firings, captions (all + gated), splits, base rates
import json, gzip, zlib, shutil, random, math, time, re
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if not os.path.isdir(BUNDLE):
    z = "/content/relabel_bundle.zip"; assert os.path.exists(z), "upload relabel_bundle.zip to /content"
    shutil.unpack_archive(z, BUNDLE)
T = np.load(f"{BUNDLE}/targets.npz")
V = T["V"]; N_V = len(V); n_spans = int(T["n_spans"])
tPS, tPT, tVL, tPA = T["span"], T["tok"], T["vlat"], T["act"].astype(np.float32)
spans = [json.loads(l) for l in gzip.open(f"{BUNDLE}/spans.jsonl.gz", "rt")]
CAPS = {json.loads(l)["latent"]: json.loads(l) for l in open(f"{BUNDLE}/relabels.jsonl")}
GATED = {int(k) for k in json.load(open(f"{BUNDLE}/labels_final.json"))}
vmap = {int(l): i for i, l in enumerate(V)}
CAP_OF = [CAPS.get(int(l), {}).get("caption", f"latent {l}") for l in V]
IS_GATED = np.array([int(l) in GATED for l in V])
Y = np.zeros((n_spans, N_V), bool); Y[tPS, tVL] = True                     # span-level firings
DOMS = ["web", "code", "cot", "math", "chat"]
dom_of = np.array([DOMS.index(s["dom"]) for s in spans])
docs = np.array([s["doc"] for s in spans])
tr_docs = set(d for d in np.unique(docs) if (zlib.crc32(str(d).encode()) % 100) < TRAIN_FRAC * 100)
tr = np.array([d in tr_docs for d in docs]); te = ~tr                      # split BY DOC, never by span
freq = Y[tr].mean(0)
order_v = np.argsort(tVL, kind="stable")                                   # latent -> rows, for witnesses later
print(f"spans {n_spans} · latents {N_V} ({IS_GATED.sum()} gated) · pairs {len(tPS)} · train/test {tr.sum()}/{te.sum()}")

spans 43669 · latents 2048 (874 gated) · pairs 9315351 · train/test 30959/12710


In [4]:
# 4 · two token views of the SAME spans + targets for each (the tokenizer confound, handled explicitly)
# view A: teacher token strings, hashed  — targets are exact (teacher fired on these very positions)
# view B: mega_tok 32k BPE ids           — targets re-aligned by char overlap with teacher tokens
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=f"{BUNDLE}/mega_tok.model")
UNI_A, UNI_B = 65536, sp.vocab_size()
def hsh(s): return zlib.crc32(s.encode()) % UNI_A
def offsets(text, toks):                                                    # walk token strings through the raw text
    out, pos = [], 0
    for t in toks:
        t = t.strip()
        j = text.find(t, pos) if t and t != "·" else -1
        out.append((j, j + len(t)) if j >= 0 else (pos, pos))
        if j >= 0: pos = j + len(t)
    return out
by_span = [[] for _ in range(n_spans)]                                      # teacher-view sparse targets per span
for s_, t_, v_, a_ in zip(tPS, tPT, tVL, tPA): by_span[s_].append((t_, v_, a_))
IDS_A, TGT_A, IDS_B, TGT_B, NTOK_B = [], [], [], [], []
for si, s in enumerate(spans):
    toks = s["toks"][:MAXLEN]
    IDS_A.append(np.array([hsh(t) for t in toks], np.int32))
    TGT_A.append([(t, v, a) for t, v, a in by_span[si] if t < MAXLEN])
    offA = offsets(s["text"], toks)
    pieces = sp.encode(s["text"], out_type=str)[:MAXLEN]
    offB = offsets(s["text"], [p.replace("\u2581", " ") for p in pieces])
    IDS_B.append(np.array(sp.encode(s["text"])[:MAXLEN], np.int32))
    tg = []
    for t, v, a in TGT_A[-1]:
        a0, a1 = offA[t]
        if a1 <= a0: continue
        for j, (b0, b1) in enumerate(offB):
            if b0 < a1 and a0 < b1: tg.append((j, v, a))                    # char overlap -> latent lands on new token
    TGT_B.append(tg); NTOK_B.append(len(IDS_B[-1]))
    if si % 8000 == 0: print(f"  aligned {si}/{n_spans}")
tok_rate = np.zeros(N_V)                                                    # token-level base rate -> log-prior bias (rung D)
n_tok_tr = sum(len(IDS_B[i]) for i in np.where(tr)[0])
for i in np.where(tr)[0]:
    for _, v, _ in TGT_B[i]: tok_rate[v] += 1
tok_rate = np.clip(tok_rate / max(n_tok_tr, 1), 1e-6, 0.5)
PRIOR = torch.tensor(np.log(tok_rate / (1 - tok_rate)), dtype=torch.float32)
print("views ready | mega tokens/span mean", round(np.mean(NTOK_B), 1), "| teacher", round(np.mean([len(x) for x in IDS_A]), 1))

  aligned 0/43669
  aligned 8000/43669
  aligned 16000/43669
  aligned 24000/43669
  aligned 32000/43669
  aligned 40000/43669
views ready | mega tokens/span mean 26.9 | teacher 27.2


In [5]:
# 5 · the golf probe (one class, ALL rungs: factorized / prior-bias / smear-gate / no-bigram) + honest metrics
def bigram(ids): return np.concatenate([[0], (36313 * ids[1:] ^ 27191 * ids[:-1]) % BIG_H]).astype(np.int64)
class Golf(nn.Module):
    def __init__(self, n_uni, fdim=None, prior=None, smear=False, use_big=True):
        super().__init__(); d = CDIM; e = fdim or d
        self.uni, self.big, self.use_big = nn.Embedding(n_uni, e), nn.Embedding(BIG_H, e), use_big
        nn.init.zeros_(self.big.weight)                                    # bigrams learn only where context pays
        if not use_big: self.big.weight.requires_grad_(False)              # rung E: table out, gate in
        self.proj = nn.Linear(e, d, bias=False) if fdim else None
        self.smear = smear                                                 # x_t += lam*sigmoid(W x_t[:12])*x_{t-1} (golf PR #1667)
        if smear:
            self.sg = nn.Linear(12, 1, bias=False); nn.init.zeros_(self.sg.weight)
            self.slam = nn.Parameter(torch.zeros(1))
        self.pos = nn.Embedding(MAXLEN, d)
        enc = nn.TransformerEncoderLayer(d, NHEAD, 4 * d, batch_first=True, dropout=0.1)
        self.enc = nn.TransformerEncoder(enc, NLAYER)
        self.head = nn.Linear(d, N_V)
        if prior is not None: self.head.bias.data = prior.clone()          # zipf-but-not-zipf: base rate lives in the bias
    def forward(self, u, b, m):
        x = self.uni(u) + (self.big(b) if self.use_big else 0)
        if self.proj is not None: x = self.proj(x)
        if self.smear:
            g = self.slam * torch.sigmoid(self.sg(x[:, 1:, :12]))
            x = torch.cat([x[:, :1], x[:, 1:] + g * x[:, :-1]], 1)
        x = x + self.pos(torch.arange(u.shape[1], device=u.device))[None]
        x = self.enc(x, src_key_padding_mask=~m)
        tok = self.head(x)                                                 # [B, L, V] per-token logits
        span = torch.logsumexp(tok.masked_fill(~m[..., None], -1e30), 1) - torch.log(m.sum(1, keepdim=True).float())
        return span, tok
def honest(S):                                                              # AP-lift over freq + z-p@5 + AUC last, per domain
    from sklearn.metrics import average_precision_score, roc_auc_score
    mu, sd = S[tr].mean(0), S[tr].std(0) + 1e-4; Z = (S - mu) / sd
    res = {}
    for tag, m in [("all", te)] + [(d, te & (dom_of == i)) for i, d in enumerate(DOMS)]:
        idx = np.where(m)[0]
        if not len(idx): continue
        top = np.argsort(-Z[idx], 1)[:, :5]
        res[tag] = {"z_p5": float(np.mean([Y[i, t].mean() for i, t in zip(idx, top)]))}
    aps, aucs = [], []
    for c in range(N_V):
        y = Y[te, c]
        if 0 < y.sum() < y.size:
            aps.append(average_precision_score(y, S[te, c]) - average_precision_score(y, np.full(y.size, freq[c])))
            aucs.append(roc_auc_score(y, S[te, c]))
    res["all"].update({"AP_lift": float(np.mean(aps)), "macroAUC": float(np.mean(aucs))})
    return res, mu, sd
print("model + metrics ready")

model + metrics ready


In [6]:
# 6 · trainer — vectorized collate + heartbeat + FIX: LR 1e-3, per-latent pos_weight (proven sae12 recipe)
LR = 1e-3
for TGT in (TGT_A, TGT_B):                                                  # idempotent: tuple lists -> arrays, once
    for i, tg in enumerate(TGT):
        if isinstance(tg, list):
            a = np.array(tg, np.float32).reshape(-1, 3) if tg else np.zeros((0, 3), np.float32)
            TGT[i] = (a[:, 0].astype(np.int64), a[:, 1].astype(np.int64), a[:, 2].copy())
_tot = sum(len(TGT_A[i][0]) for i in range(n_spans))
print(f"targets sane? TGT_A pairs {_tot} ({_tot/n_spans:.0f}/span) · active latents/span {Y.sum(1).mean():.0f}")
_pos = np.maximum(tok_rate * n_tok_tr, 1)
PW = torch.tensor(np.clip((n_tok_tr - _pos) / _pos, 1, 25), dtype=torch.float32)
def batches(idx, IDS, TGT, shuffle=True):
    idx = list(idx)
    if shuffle: random.shuffle(idx)
    idx.sort(key=lambda i: len(IDS[i]))
    for b0 in range(0, len(idx), BS):
        ch = idx[b0:b0 + BS]; L = max(len(IDS[i]) for i in ch)
        u = torch.zeros(len(ch), L, dtype=torch.long); m = torch.zeros(len(ch), L, dtype=torch.bool)
        bg = torch.zeros(len(ch), L, dtype=torch.long)
        yb = torch.zeros(len(ch), L, N_V); qa = torch.zeros(len(ch), L, N_V)
        for r, i in enumerate(ch):
            n = len(IDS[i]); u[r, :n] = torch.tensor(np.asarray(IDS[i]), dtype=torch.long); m[r, :n] = True
            bg[r, :n] = torch.tensor(bigram(np.asarray(IDS[i])), dtype=torch.long)
            tt, vv, aa = TGT[i]; k = tt < L
            ti, vi = torch.from_numpy(tt[k]), torch.from_numpy(vv[k])
            yb[r, ti, vi] = 1.0
            qa[r].index_put_((ti, vi), torch.from_numpy(aa[k]), accumulate=True)
        yield ch, u, bg, m, yb, qa
def score_all(model, IDS, TGT):
    model.eval(); S = np.zeros((n_spans, N_V), np.float32)
    with torch.no_grad():
        for ch, u, b, m, _, _ in batches(range(n_spans), IDS, TGT, shuffle=False):
            s, _ = model(u.to(DEV), b.to(DEV), m.to(DEV)); S[ch] = s.float().cpu().numpy()
    return S
def train_rung(tag, IDS, TGT, n_uni, fdim=None, prior=None, soft=False, smear=False, use_big=True, table_wd=None):
    model = Golf(n_uni, fdim, prior, smear, use_big).to(DEV)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_batch = int(tr.sum()) // BS + 1
    print(f"{tag}: {n_par/1e6:.2f}M trainable · {n_batch} batches/epoch · {EPOCHS} epochs · lr {LR}")
    if table_wd is None:
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
    else:                                                   # deliberate decay on the tables only (rare rows fade -> compress better)
        tabs = [model.uni.weight] + ([model.big.weight] if model.big.weight.requires_grad else [])
        tset = {id(p) for p in tabs}
        rest = [p for p in model.parameters() if p.requires_grad and id(p) not in tset]
        opt = torch.optim.AdamW([{"params": tabs, "weight_decay": table_wd},
                                 {"params": rest, "weight_decay": 1e-4}], lr=LR)
    pw = PW.to(DEV)
    best = (-1, None)
    for ep in range(1, EPOCHS + 1):
        model.train(); ls, nb, t0 = 0.0, 0, time.time()
        for _, u, b, m, yb, qa in batches(np.where(tr)[0], IDS, TGT):
            u, b, m, yb, qa = u.to(DEV), b.to(DEV), m.to(DEV), yb.to(DEV), qa.to(DEV)
            _, tok = model(u, b, m)
            loss = F.binary_cross_entropy_with_logits(tok[m], yb[m], pos_weight=pw)
            if soft:
                fire = m & (qa.sum(-1) > 0)
                if fire.any():
                    q = qa[fire] / qa[fire].sum(-1, keepdim=True)
                    loss = loss + SOFT_W * F.kl_div(F.log_softmax(tok[fire] / TEMP, -1), q, reduction="batchmean")
            opt.zero_grad(); loss.backward(); opt.step()
            ls += float(loss.detach()); nb += 1
            if ep == 1 and nb == 20:
                print(f"  {tag} alive · 20/{n_batch} batches · {(time.time()-t0)/20:.2f}s/batch "
                      f"-> ~{(time.time()-t0)/20*n_batch/60:.1f} min/epoch")
        line = f"  {tag} ep{ep} loss {ls/max(nb,1):.4f} · {time.time()-t0:.0f}s"
        if ep % 3 == 0 or ep == EPOCHS:
            S = score_all(model, IDS, TGT); r, _, _ = honest(S)
            zp = r["all"]["z_p5"]
            if zp > best[0]: best = (zp, {k: v.cpu().clone() for k, v in model.state_dict().items()})
            line += f" · z-p@5 {zp:.3f} (best {best[0]:.3f})"
        print(line)
    model.load_state_dict(best[1]); S = score_all(model, IDS, TGT)
    res, mu, sd = honest(S)
    print(f"{tag} DONE: {n_par/1e6:.2f}M · " + " · ".join(f"{k} {v:.3f}" for k, v in res["all"].items())
          + " | per-dom z-p@5 " + " ".join(f"{d}:{res[d]['z_p5']:.2f}" for d in DOMS if d in res))
    return {"model": model, "S": S, "res": res, "mu": mu, "sd": sd, "params": n_par}
print("trainer ready (lr 1e-3 · per-latent pos_weight)")


targets sane? TGT_A pairs 9315351 (213/span) · active latents/span 138
trainer ready (lr 1e-3 · per-latent pos_weight)


In [ ]:
# 7 · RUNG A — old nano recipe, teacher-token hashes, new data + dictionary (the baseline everything answers to)
R = {}
R["A"] = train_rung("A", IDS_A, TGT_A, UNI_A)

A: 23.11M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  A alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  A ep1 loss 0.2396 · 13s
  A ep2 loss 0.1573 · 13s
  A ep3 loss 0.1355 · 13s · z-p@5 0.911 (best 0.911)
  A ep4 loss 0.1239 · 13s
  A ep5 loss 0.1163 · 13s
  A ep6 loss 0.1105 · 13s · z-p@5 0.935 (best 0.935)
  A ep7 loss 0.1061 · 13s
  A ep8 loss 0.1023 · 13s
  A ep9 loss 0.0991 · 13s · z-p@5 0.944 (best 0.944)
  A ep10 loss 0.0963 · 13s
  A ep11 loss 0.0939 · 13s
  A ep12 loss 0.0917 · 13s · z-p@5 0.950 (best 0.950)
  A ep13 loss 0.0898 · 13s
  A ep14 loss 0.0879 · 14s
  A ep15 loss 0.0862 · 13s · z-p@5 0.953 (best 0.953)
A DONE: 23.11M · z_p5 0.953 · AP_lift 0.483 · macroAUC 0.900 | per-dom z-p@5 web:0.92 code:0.95 cot:0.97 math:0.97 chat:0.98


In [ ]:
# 8 · RUNG B — one change: the new 32k tokenizer (exact 32k table, no unigram hashing; targets char-aligned)
R["B"] = train_rung("B", IDS_B, TGT_B, UNI_B)

B: 14.72M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  B alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  B ep1 loss 0.2501 · 12s
  B ep2 loss 0.1811 · 12s
  B ep3 loss 0.1570 · 12s · z-p@5 0.890 (best 0.890)
  B ep4 loss 0.1428 · 12s
  B ep5 loss 0.1332 · 12s
  B ep6 loss 0.1263 · 12s · z-p@5 0.917 (best 0.917)
  B ep7 loss 0.1208 · 12s
  B ep8 loss 0.1163 · 12s
  B ep9 loss 0.1125 · 12s · z-p@5 0.929 (best 0.929)
  B ep10 loss 0.1092 · 12s
  B ep11 loss 0.1063 · 12s
  B ep12 loss 0.1038 · 12s · z-p@5 0.935 (best 0.935)
  B ep13 loss 0.1014 · 12s
  B ep14 loss 0.0993 · 12s
  B ep15 loss 0.0974 · 12s · z-p@5 0.940 (best 0.940)
B DONE: 14.72M · z_p5 0.940 · AP_lift 0.455 · macroAUC 0.891 | per-dom z-p@5 web:0.90 code:0.94 cot:0.96 math:0.95 chat:0.98


In [ ]:
# 9 · RUNG C — one change: factorized tables (d=64 -> proj 256); the size lever
R["C"] = train_rung("C", IDS_B, TGT_B, UNI_B, fdim=FDIM)

C: 5.30M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  C alive · 20/323 batches · 0.02s/batch -> ~0.1 min/epoch
  C ep1 loss 0.2571 · 11s
  C ep2 loss 0.1912 · 11s
  C ep3 loss 0.1684 · 11s · z-p@5 0.858 (best 0.858)
  C ep4 loss 0.1543 · 11s
  C ep5 loss 0.1445 · 11s
  C ep6 loss 0.1373 · 11s · z-p@5 0.900 (best 0.900)
  C ep7 loss 0.1318 · 11s
  C ep8 loss 0.1273 · 11s
  C ep9 loss 0.1236 · 11s · z-p@5 0.917 (best 0.917)
  C ep10 loss 0.1206 · 11s
  C ep11 loss 0.1180 · 11s
  C ep12 loss 0.1154 · 11s · z-p@5 0.929 (best 0.929)
  C ep13 loss 0.1133 · 11s
  C ep14 loss 0.1114 · 11s
  C ep15 loss 0.1096 · 11s · z-p@5 0.935 (best 0.935)
C DONE: 5.30M · z_p5 0.935 · AP_lift 0.440 · macroAUC 0.885 | per-dom z-p@5 web:0.89 code:0.94 cot:0.96 math:0.94 chat:0.98


In [ ]:
# 10 · RUNG D — the entropy golf: log-prior bias head + soft distillation on teacher act distributions
R["D"] = train_rung("D", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True)

D: 5.30M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  D alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  D ep1 loss 1.4440 · 12s
  D ep2 loss 1.0962 · 12s
  D ep3 loss 0.9823 · 12s · z-p@5 0.929 (best 0.929)
  D ep4 loss 0.9129 · 12s
  D ep5 loss 0.8662 · 12s
  D ep6 loss 0.8317 · 12s · z-p@5 0.939 (best 0.939)
  D ep7 loss 0.8050 · 12s
  D ep8 loss 0.7830 · 12s
  D ep9 loss 0.7649 · 12s · z-p@5 0.943 (best 0.943)
  D ep10 loss 0.7493 · 12s
  D ep11 loss 0.7360 · 12s
  D ep12 loss 0.7241 · 12s · z-p@5 0.947 (best 0.947)
  D ep13 loss 0.7133 · 12s
  D ep14 loss 0.7041 · 12s
  D ep15 loss 0.6956 · 12s · z-p@5 0.948 (best 0.948)
D DONE: 5.30M · z_p5 0.948 · AP_lift 0.470 · macroAUC 0.889 | per-dom z-p@5 web:0.93 code:0.93 cot:0.97 math:0.95 chat:0.98


In [ ]:
# 10b · RUNG E — smear gate REPLACES the bigram table (13 params vs ~1M): compute the context, don't memorize it
R["E"] = train_rung("E", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True, smear=True, use_big=False)

E: 4.25M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  E alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  E ep1 loss 1.4461 · 12s
  E ep2 loss 1.1013 · 12s
  E ep3 loss 0.9911 · 12s · z-p@5 0.930 (best 0.930)
  E ep4 loss 0.9257 · 12s
  E ep5 loss 0.8814 · 12s
  E ep6 loss 0.8489 · 12s · z-p@5 0.937 (best 0.937)
  E ep7 loss 0.8238 · 12s
  E ep8 loss 0.8034 · 12s
  E ep9 loss 0.7864 · 12s · z-p@5 0.943 (best 0.943)
  E ep10 loss 0.7722 · 12s
  E ep11 loss 0.7604 · 12s
  E ep12 loss 0.7493 · 12s · z-p@5 0.945 (best 0.945)
  E ep13 loss 0.7400 · 12s
  E ep14 loss 0.7314 · 12s
  E ep15 loss 0.7242 · 12s · z-p@5 0.947 (best 0.947)
E DONE: 4.25M · z_p5 0.947 · AP_lift 0.467 · macroAUC 0.888 | per-dom z-p@5 web:0.93 code:0.93 cot:0.97 math:0.95 chat:0.98


In [ ]:
# 6b · FLOORS — every later number is read as distance from these (free, instant)
p = tok_rate
w = PW.cpu().numpy()
phat = w * p / (w * p + 1 - p)                       # optimal constant guess under pos_weight
LOSS_FLOOR = float((-(w * p * np.log(phat + 1e-12) + (1 - p) * np.log(1 - phat + 1e-12))).mean())
DENS_FLOOR = float(Y[te].mean())                     # random-5 p@5 floor
FREQ5_FLOOR = float(Y[te][:, np.argsort(-freq)[:5]].mean())   # "always show the 5 most frequent" floor
print(f"active latents/span (test): {Y[te].sum(1).mean():.0f} / {N_V}")
print(f"loss floor (ignore-the-text): {LOSS_FLOOR:.4f} — a healthy rung ends well below")
print(f"p@5 floors: random {DENS_FLOOR:.3f} · frequency-5 {FREQ5_FLOOR:.3f} — z-p@5 is read as lift over frequency-5")

active latents/span (test): 138 / 2048
loss floor (ignore-the-text): 0.3034 — a healthy rung ends well below
p@5 floors: random 0.067 · frequency-5 0.297 — z-p@5 is read as lift over frequency-5


In [ ]:
# 11 · the ladder, in one table (params · MB · honest metrics · per-domain) — each row differs by ONE variable
print(f"{'rung':4s} {'params':>8s} {'MBfp32':>7s} {'z-p@5':>6s} {'APlift':>7s} {'AUC':>6s}  per-domain z-p@5")
for k in [k for k in "ABCDE" if k in R]:
    r = R[k]["res"]["all"]; p = R[k]["params"]
    dom = " ".join(f"{d}:{R[k]['res'][d]['z_p5']:.2f}" for d in DOMS if d in R[k]["res"])
    print(f"{k:4s} {p/1e6:7.2f}M {p*4/1e6:6.1f}M {r['z_p5']:6.3f} {r['AP_lift']:+7.3f} {r['macroAUC']:6.3f}  {dom}")
print("read: B-A = tokenizer · C-B = factorization cost · D-C = entropy golf gain · E-D = smear-for-table trade")
BEST_K = max((k for k in "DE" if k in R), key=lambda k: R[k]["res"]["all"]["z_p5"])
print(f"BEST entropy rung = {BEST_K} (used by the surprise lens, export, judge and teacher-check cells)")

rung   params  MBfp32  z-p@5  APlift    AUC  per-domain z-p@5
A      23.11M   92.4M  0.953  +0.483  0.900  web:0.92 code:0.95 cot:0.97 math:0.97 chat:0.98
B      14.72M   58.9M  0.940  +0.455  0.891  web:0.90 code:0.94 cot:0.96 math:0.95 chat:0.98
C       5.30M   21.2M  0.935  +0.440  0.885  web:0.89 code:0.94 cot:0.96 math:0.94 chat:0.98
D       5.30M   21.2M  0.948  +0.470  0.889  web:0.93 code:0.93 cot:0.97 math:0.95 chat:0.98
E       4.25M   17.0M  0.947  +0.467  0.888  web:0.93 code:0.93 cot:0.97 math:0.95 chat:0.98
read: B-A = tokenizer · C-B = factorization cost · D-C = entropy golf gain · E-D = smear-for-table trade
BEST entropy rung = D (used by the surprise lens, export, judge and teacher-check cells)


In [ ]:
# 12 · SURPRISE lens — per-token concept entropy + adjacent-token KL, VALIDATED against the teacher's own boundaries
best = R[BEST_K]["model"]; best.eval(); print("surprise lens on rung", BEST_K)
def curves(tok_logits, n):
    p = F.softmax(tok_logits[:n] / TEMP, -1)
    H = (-p * (p + 1e-9).log()).sum(-1)
    kl = torch.zeros(n)
    kl[1:] = (p[1:] * ((p[1:] + 1e-9).log() - (p[:-1] + 1e-9).log())).sum(-1)
    return H.cpu().numpy(), kl.cpu().numpy()
hits, base = [], []
test_idx = [i for i in np.where(te)[0] if len(TGT_B[i][0]) > 8 and len(IDS_B[i]) >= 16]
print("eligible test spans:", len(test_idx))
for i in test_idx[:400]:
    n = len(IDS_B[i]); qa = np.zeros((n, N_V), np.float32)
    tt, vv, aa = TGT_B[i]; k = tt < n
    np.add.at(qa, (tt[k], vv[k]), aa[k])
    live = qa.sum(1) > 0
    q = qa / np.maximum(qa.sum(1, keepdims=True), 1e-9)
    tkl = np.zeros(n); tkl[1:] = (q[1:] * (np.log(q[1:] + 1e-9) - np.log(q[:-1] + 1e-9))).sum(1)
    tkl[~live] = 0
    u = torch.tensor(np.asarray(IDS_B[i]), dtype=torch.long)[None].to(DEV)
    b = torch.tensor(bigram(np.asarray(IDS_B[i])))[None].to(DEV); m = torch.ones_like(u, dtype=torch.bool)
    with torch.no_grad(): _, tok = best(u, b, m)
    _, skl = curves(tok[0], n)
    kk = max(2, int(0.15 * n))
    sb, tb = set(np.argsort(-skl)[:kk]), set(np.argsort(-tkl)[:kk])
    hits.append(len(sb & tb) / kk); base.append(kk / n)
print(f"boundary agreement (student KL peaks vs TEACHER label-set changes): p@15% {np.mean(hits):.3f} "
      f"vs chance {np.mean(base):.3f} — {np.mean(hits)/np.mean(base):.1f}x")
i = test_idx[0]; n = len(IDS_B[i])
u = torch.tensor(np.asarray(IDS_B[i]), dtype=torch.long)[None].to(DEV)
b = torch.tensor(bigram(np.asarray(IDS_B[i])))[None].to(DEV)
with torch.no_grad(): _, tok = best(u, b, torch.ones_like(u, dtype=torch.bool))
H, kl = curves(tok[0], n); cut = np.sort(kl)[-max(2, int(0.15 * n))]
pieces = sp.encode(spans[i]["text"], out_type=str)[:n]
print("".join(("⎇" if kl[j] >= cut and j else "") + pc.replace("▁", " ") for j, pc in enumerate(pieces))[:400])


surprise lens on rung D
eligible test spans: 9747
boundary agreement (student KL peaks vs TEACHER label-set changes): p@15% 0.247 vs chance 0.132 — 1.9x
 *sigh* Fundamentalist community,⎇ let me pass on⎇ some advice to you I learned⎇ from the atheistic community:


In [ ]:
# 13 · export the winner: weights + captions (gated flag) + prior + per-domain z calibration -> golf_bundle.zip
OUTG = "/content/golf_out"; os.makedirs(OUTG, exist_ok=True)
mu_dom = {d: R[BEST_K]["S"][tr & (dom_of == i)].mean(0).tolist() for i, d in enumerate(DOMS)}
sd_dom = {d: (R[BEST_K]["S"][tr & (dom_of == i)].std(0) + 1e-4).tolist() for i, d in enumerate(DOMS)}
torch.save({"state_dict": R[BEST_K]["model"].state_dict(), "rung": BEST_K,
            "cfg": {"UNI": UNI_B, "BIG_H": BIG_H, "CDIM": CDIM, "FDIM": FDIM, "NHEAD": NHEAD, "NLAYER": NLAYER,
                    "MAXLEN": MAXLEN, "TEMP": TEMP, "smear": BEST_K == "E", "use_big": BEST_K != "E"},
            "V": V.tolist(), "captions": CAP_OF, "gated": IS_GATED.tolist(), "prior": PRIOR.tolist(),
            "mu": R[BEST_K]["mu"].tolist(), "sd": R[BEST_K]["sd"].tolist(), "mu_dom": mu_dom, "sd_dom": sd_dom},
           f"{OUTG}/mega_golf_probe.pth")
shutil.copy(f"{BUNDLE}/mega_tok.model", OUTG)
json.dump({k: {**R[k]["res"]["all"], "params": R[k]["params"]} for k in R}, open(f"{OUTG}/ladder.json", "w"), indent=1)
shutil.make_archive("/content/golf_bundle", "zip", OUTG)
print("saved:", os.listdir(OUTG), f"| probe {os.path.getsize(f'{OUTG}/mega_golf_probe.pth')/1e6:.1f}MB fp32")

saved: ['ladder.json', 'mega_tok.model', 'mega_golf_probe.pth'] | probe 21.6MB fp32


In [ ]:
# 14 · LLM JUDGE — rung A vs rung D on held-out spans: top-5 labels + witnesses (+ D's boundaries); who is more useful?
import requests, threading
# assert OPENROUTER_KEY.startswith("sk-or-"), "set OPENROUTER_KEY to run the judge"
OPENROUTER_KEY = os.environ.get("OPENROUTER_KEY","")  # set your key in the environment
def ask(prompt, max_tok=400):
    for a in range(3):
        try:
            r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": "Bearer " + OPENROUTER_KEY},
                json={"model": JUDGE_MODEL, "messages": [{"role": "user", "content": prompt}],
                      "reasoning": {"enabled": False}, "temperature": 0.1, "max_tokens": max_tok}, timeout=90)
            t = r.json()["choices"][0]["message"]["content"]
            return json.loads(t[t.index("{"): t.rindex("}") + 1])
        except Exception: time.sleep(2 * a + 1)
    return None
def readout(rk, i, boundaries=False):
    IDS = IDS_A if rk == "A" else IDS_B; ids = IDS[i]
    u = torch.tensor(np.asarray(ids), dtype=torch.long)[None].to(DEV)
    b = torch.tensor(bigram(np.asarray(ids)))[None].to(DEV); m = torch.ones_like(u, dtype=torch.bool)
    with torch.no_grad(): s, tok = R[rk]["model"](u, b, m)
    z = (s[0].cpu().numpy() - R[rk]["mu"]) / R[rk]["sd"]
    toks = spans[i]["toks"] if rk == "A" else [p.replace("\u2581", " ") for p in sp.encode(spans[i]["text"], out_type=str)[:len(ids)]]
    lines = []
    for c in np.argsort(-z)[:5]:
        wit = [toks[t] for t in torch.argsort(-tok[0, :, c]).cpu().numpy()[:3] if t < len(toks)]
        lines.append(f"z={z[c]:+.2f} [{'G' if IS_GATED[c] else 'u'}] {CAP_OF[c][:90]} | witnesses: {', '.join(wit)}")
    if boundaries:
        _, kl = curves(tok[0], len(ids)); cut = np.sort(kl)[-max(2, int(0.15 * len(ids)))]
        bts = [toks[j] for j in range(1, len(ids)) if kl[j] >= cut][:8]
        lines.append("boundary tokens: " + ", ".join(bts))
    return "\n".join(lines)
J_PROMPT = ("TEXT:\n{txt}\n\nMONITOR OUTPUT (concept labels + witness tokens{extra}):\n{out}\n\n"
            "Judge each label: good (describes the text, witnesses support it), weak (partial), noise (wrong). "
            '{bq}Reply ONLY JSON: {{"good": <n>, "weak": <n>, "noise": <n>{bk}}}')
rng = np.random.default_rng(SEED)
picks = [int(rng.choice(np.where(te & (dom_of == i))[0])) for i in range(len(DOMS)) for _ in (0, 1)][:N_JUDGE]
tally = {"A": [0, 0, 0], BEST_K: [0, 0, 0]}; bd = []
for i in picks:
    for rk in ("A", BEST_K):
        extra = " + decision-boundary tokens" if rk != "A" else ""
        bq = "Also rate whether the boundary tokens mark real transitions (0-5). " if rk != "A" else ""
        bk = ', "boundaries": <0-5>' if rk != "A" else ""
        res = ask(J_PROMPT.format(txt=spans[i]["text"][:600], out=readout(rk, i, rk != "A"), extra=extra, bq=bq, bk=bk))
        if not res: continue
        for j, k in enumerate(("good", "weak", "noise")): tally[rk][j] += int(res.get(k, 0) or 0)
        if rk != "A" and "boundaries" in res: bd.append(int(res["boundaries"]))
for rk in ("A", BEST_K):
    g, w, x = tally[rk]; print(f"rung {rk}: {g} good · {w} weak · {x} noise  ({g/max(g+w+x,1):.0%} good)")
print("boundary usefulness (judge, 0-5): mean", round(np.mean(bd), 2) if bd else "n/a")

rung A: 20 good · 16 weak · 4 noise  (50% good)
rung D: 20 good · 9 weak · 4 noise  (61% good)
boundary usefulness (judge, 0-5): mean 3.38


In [ ]:
# 15 · OPTIONAL teacher agreement spot-check (A100 only; settles "is the noise mine or the SAE's")
if LOAD_TEACHER:
    !pip -q install transformer-lens sae-lens
    from transformer_lens import HookedTransformer
    from sae_lens import SAE
    tm = HookedTransformer.from_pretrained("gemma-2-9b", device=DEV, dtype=torch.bfloat16)
    _r = SAE.from_pretrained("gemma-scope-9b-pt-res-canonical", "layer_20/width_16k/canonical", device=DEV)
    tsae = (_r[0] if isinstance(_r, tuple) else _r).to(torch.bfloat16)
    hook = getattr(tsae.cfg, "hook_name", None) or tsae.cfg.metadata.hook_name
    lut = np.full(16384, -1); lut[V] = np.arange(N_V)
    over = []
    for i in [int(x) for x in np.where(te)[0][:12]]:
        ids = tm.to_tokens(spans[i]["text"])[:, :MAXLEN]
        _, c = tm.run_with_cache(ids, names_filter=hook, return_type=None)
        f = tsae.encode(c[hook]).float()[0].max(0).values.cpu().numpy()    # teacher span score = max act
        tz = np.full(N_V, -9.0); ok_ = lut[np.arange(16384)] >= 0
        tz[lut[ok_]] = f[ok_]
        u = torch.tensor(np.asarray(IDS_B[i]), dtype=torch.long)[None].to(DEV)
        b = torch.tensor(bigram(np.asarray(IDS_B[i])))[None].to(DEV)
        with torch.no_grad(): s, _ = R[BEST_K]["model"](u, b, torch.ones_like(u, dtype=torch.bool))
        z = (s[0].cpu().numpy() - R[BEST_K]["mu"]) / R[BEST_K]["sd"]
        a, bset = set(np.argsort(-z)[:10]), set(np.argsort(-tz)[:10])
        over.append(len(a & bset) / 10)
    print("student top-10 vs teacher top-10 overlap:", round(float(np.mean(over)), 3))
else:
    print("skipped (set LOAD_TEACHER=True on an A100 to run)")

skipped (set LOAD_TEACHER=True on an A100 to run)


In [ ]:
# 16 · FREE QUANTS — fake-quantize the winner, re-score honest metrics (no retraining)
import copy
def fq(w, bits):
    if bits >= 16: return w.half().float()
    qmax = 2 ** (bits - 1) - 1
    s = w.abs().amax(dim=-1, keepdim=True) / qmax + 1e-12
    return (w / s).round().clamp(-qmax, qmax) * s
print(f"rung {BEST_K} fp32 baseline: z-p@5 {R[BEST_K]['res']['all']['z_p5']:.3f} · {R[BEST_K]['params']*4/1e6:.1f}MB")
for bits in [16, 8, 6, 4]:
    m2 = copy.deepcopy(R[BEST_K]["model"]).to(DEV)
    with torch.no_grad():
        for _, p_ in m2.named_parameters(): p_.copy_(fq(p_.data, bits))
    S2 = score_all(m2, IDS_B, TGT_B); r2, _, _ = honest(S2)
    print(f"{bits:>2}-bit ~{R[BEST_K]['params']*bits/8/1e6:5.1f}MB | z-p@5 {r2['all']['z_p5']:.3f} | AP-lift {r2['all']['AP_lift']:+.3f}")


rung D fp32 baseline: z-p@5 0.948 · 21.2MB
16-bit ~ 10.6MB | z-p@5 0.947 | AP-lift +0.470
 8-bit ~  5.3MB | z-p@5 0.947 | AP-lift +0.469
 6-bit ~  4.0MB | z-p@5 0.947 | AP-lift +0.469
 4-bit ~  2.6MB | z-p@5 0.943 | AP-lift +0.459


In [ ]:
# 17 · FINAL TRAIN — winner recipe, not undertrained: 40 epochs, cosine LR, best checkpoint kept
EPOCHS_F = 40
model = Golf(UNI_B, fdim=FDIM, prior=PRIOR).to(DEV)            # rung D recipe (add smear=True, use_big=False for E)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
n_batch = int(tr.sum()) // BS + 1
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_F * n_batch)
pw = PW.to(DEV); best = (-1, None)
for ep in range(1, EPOCHS_F + 1):
    model.train(); ls, nb, t0 = 0.0, 0, time.time()
    for _, u, b, m, yb, qa in batches(np.where(tr)[0], IDS_B, TGT_B):
        u, b, m, yb, qa = u.to(DEV), b.to(DEV), m.to(DEV), yb.to(DEV), qa.to(DEV)
        _, tok = model(u, b, m)
        loss = F.binary_cross_entropy_with_logits(tok[m], yb[m], pos_weight=pw)
        fire = m & (qa.sum(-1) > 0)
        if fire.any():
            q = qa[fire] / qa[fire].sum(-1, keepdim=True)
            loss = loss + SOFT_W * F.kl_div(F.log_softmax(tok[fire] / TEMP, -1), q, reduction="batchmean")
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        ls += float(loss.detach()); nb += 1
    line = f"  F ep{ep} loss {ls/nb:.4f} · {time.time()-t0:.0f}s · lr {sched.get_last_lr()[0]:.1e}"
    if ep % 2 == 0 or ep == EPOCHS_F:
        S = score_all(model, IDS_B, TGT_B); r, _, _ = honest(S)
        zp = r["all"]["z_p5"]
        if zp > best[0]: best = (zp, {k: v.cpu().clone() for k, v in model.state_dict().items()})
        line += f" · z-p@5 {zp:.3f} (best {best[0]:.3f})"
    print(line)
model.load_state_dict(best[1]); S = score_all(model, IDS_B, TGT_B)
res, mu, sd = honest(S)
R["F"] = {"model": model, "S": S, "res": res, "mu": mu, "sd": sd,
          "params": sum(p.numel() for p in model.parameters() if p.requires_grad)}
BEST_K = "F"
print("F DONE: " + " · ".join(f"{k} {v:.3f}" for k, v in res["all"].items())
      + " | per-dom " + " ".join(f"{d}:{res[d]['z_p5']:.2f}" for d in DOMS if d in res))


  F ep1 loss 1.4392 · 12s · lr 1.0e-03
  F ep2 loss 1.0988 · 12s · lr 9.9e-04 · z-p@5 0.918 (best 0.918)
  F ep3 loss 0.9849 · 12s · lr 9.9e-04
  F ep4 loss 0.9163 · 12s · lr 9.8e-04 · z-p@5 0.936 (best 0.936)
  F ep5 loss 0.8695 · 12s · lr 9.6e-04
  F ep6 loss 0.8345 · 12s · lr 9.5e-04 · z-p@5 0.940 (best 0.940)
  F ep7 loss 0.8071 · 12s · lr 9.3e-04
  F ep8 loss 0.7850 · 12s · lr 9.0e-04 · z-p@5 0.942 (best 0.942)
  F ep9 loss 0.7663 · 12s · lr 8.8e-04
  F ep10 loss 0.7503 · 12s · lr 8.5e-04 · z-p@5 0.944 (best 0.944)
  F ep11 loss 0.7365 · 12s · lr 8.2e-04
  F ep12 loss 0.7249 · 12s · lr 7.9e-04 · z-p@5 0.945 (best 0.945)
  F ep13 loss 0.7141 · 12s · lr 7.6e-04
  F ep14 loss 0.7043 · 12s · lr 7.3e-04 · z-p@5 0.946 (best 0.946)
  F ep15 loss 0.6951 · 12s · lr 6.9e-04
  F ep16 loss 0.6874 · 12s · lr 6.5e-04 · z-p@5 0.946 (best 0.946)
  F ep17 loss 0.6797 · 12s · lr 6.2e-04
  F ep18 loss 0.6727 · 12s · lr 5.8e-04 · z-p@5 0.948 (best 0.948)
  F ep19 loss 0.6667 · 12s · lr 5.4e-04
  F ep

In [ ]:
# 18 · EYEBALL — span -> latents, human-readable (exactly what the judge saw)
for d in DOMS:
    cand = np.where(te & (dom_of == DOMS.index(d)))[0]
    i = int(cand[min(3, len(cand) - 1)])
    print(f"—— [{d}] {spans[i]['text'][:220]}")
    print(readout(BEST_K, i, True))
    print()


—— [web] I wont use it again. I really do like 'Inner Life', though, and would love to use it in classroom presentations, from the BioVisions site, if that is acceptable."
z=+5.11 [G] Comparing things using 'like' to introduce examples or analogies. | witnesses:  like,  really, In
z=+4.02 [G] The latent detects hypothetical or conditional statements, often using 'would'. | witnesses:  would,  do,  to
z=+4.01 [G] Detects the use of 'do' or 'does' as an auxiliary verb, often for emphasis or negation. | witnesses:  do,  w, ner
z=+3.68 [G] The latent detects the phrase 'do' followed by a pronoun or article, often indicating an a | witnesses:  do,  it,  it
z=+3.67 [G] This latent detects quoted strings, especially those ending with punctuation. | witnesses: .", ',,  though
boundary tokens:  use,  ',  would,  from,  if, ."

—— [code] from tardis.tardis_portal.shortcuts import get_experiment_referer
z=+4.70 [G] Python code that retrieves data or objects using a 'get' function. | witnesses:  g

In [ ]:
# 19 · EXPORT + DOWNLOAD the final probe: fp32 + real int8 pack + tokenizer + captions -> golf_final.zip
import pickle, gzip
OUTG = "/content/golf_final"; os.makedirs(OUTG, exist_ok=True)
mdl = R[BEST_K]["model"].cpu().eval()
mu_dom = {d: R[BEST_K]["S"][tr & (dom_of == i)].mean(0).tolist() for i, d in enumerate(DOMS)}
sd_dom = {d: (R[BEST_K]["S"][tr & (dom_of == i)].std(0) + 1e-4).tolist() for i, d in enumerate(DOMS)}
meta = {"rung": BEST_K,
        "cfg": {"UNI": UNI_B, "BIG_H": BIG_H, "CDIM": CDIM, "FDIM": FDIM, "NHEAD": NHEAD, "NLAYER": NLAYER,
                "MAXLEN": MAXLEN, "TEMP": TEMP, "smear": hasattr(mdl, "sg"), "use_big": mdl.use_big},
        "V": V.tolist(), "captions": CAP_OF, "gated": IS_GATED.tolist(), "prior": PRIOR.tolist(),
        "mu": R[BEST_K]["mu"].tolist(), "sd": R[BEST_K]["sd"].tolist(), "mu_dom": mu_dom, "sd_dom": sd_dom,
        "metrics": R[BEST_K]["res"]["all"]}
torch.save({"state_dict": mdl.state_dict(), **meta}, f"{OUTG}/mega_golf_probe.pth")
pack = {"meta": meta, "q": {}}                              # real int8: per-row scale + int8 weights (measured lossless)
for k, wv in mdl.state_dict().items():
    w = wv.float()
    if w.ndim >= 2:
        s = w.abs().amax(-1, keepdim=True) / 127 + 1e-12
        pack["q"][k] = ("i8", (w / s).round().to(torch.int8).numpy(), s.squeeze(-1).half().numpy())
    else:
        pack["q"][k] = ("f16", w.half().numpy(), None)
with gzip.open(f"{OUTG}/mega_golf_probe_int8.pkl.gz", "wb") as f: pickle.dump(pack, f, 4)
shutil.copy(f"{BUNDLE}/mega_tok.model", OUTG)
json.dump({k: {**R[k]["res"]["all"], "params": R[k]["params"]} for k in R}, open(f"{OUTG}/ladder.json", "w"), indent=1)
shutil.make_archive("/content/golf_final", "zip", OUTG)
for f_ in sorted(os.listdir(OUTG)):
    print(f"  {f_:32s} {os.path.getsize(f'{OUTG}/{f_}')/1e6:6.2f} MB")
print("zip:", round(os.path.getsize("/content/golf_final.zip")/1e6, 2), "MB")
mdl.to(DEV)
from google.colab import files; files.download("/content/golf_final.zip")


  ladder.json                        0.00 MB
  mega_golf_probe.pth               21.64 MB
  mega_golf_probe_int8.pkl.gz        5.29 MB
  mega_tok.model                     0.76 MB
zip: 25.47 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Lab 2 — inspect · prune · soup · decay (needs only cells 1–6 + the two bundles; NO rung retraining)
Upload `relabel_bundle.zip` AND `golf_final.zip` to `/content`. Run cells 1–6 to rebuild the data views, then these.

In [7]:
# 20 · load the SHIPPED probe as rung F (no retraining)
GF = "/content/golf_final"
if not os.path.isdir(GF) and os.path.exists("/content/golf_final.zip"): shutil.unpack_archive("/content/golf_final.zip", GF)
ck = torch.load(f"{GF}/mega_golf_probe.pth", map_location="cpu", weights_only=False)
assert ck["cfg"]["UNI"] == UNI_B, "tokenizer/bundle mismatch — use the same relabel bundle this probe was trained from"
mF = Golf(UNI_B, fdim=ck["cfg"]["FDIM"], prior=torch.tensor(ck["prior"], dtype=torch.float32)).to(DEV)
mF.load_state_dict(ck["state_dict"]); mF.eval()
S_F = score_all(mF, IDS_B, TGT_B); resF, muF, sdF = honest(S_F)
try: R
except NameError: R = {}
R["F"] = {"model": mF, "S": S_F, "res": resF, "mu": muF, "sd": sdF,
          "params": sum(p.numel() for p in mF.parameters() if p.requires_grad)}
BEST_K = "F"
OUTG = "/content/golf_out2"; os.makedirs(OUTG, exist_ok=True)
print("F loaded from disk: " + " · ".join(f"{k} {v:.3f}" for k, v in resF["all"].items()))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


F loaded from disk: z_p5 0.956 · AP_lift 0.488 · macroAUC 0.896


In [8]:
# 21 · INSPECT the tables — power law? dead rows? how much bigram did the model actually buy?
W = R["F"]["model"].uni.weight.detach().cpu(); WB = R["F"]["model"].big.weight.detach().cpu()
norms = W.norm(dim=1).numpy(); bnorms = WB.norm(dim=1).numpy()
tok_freq = np.zeros(UNI_B, np.int64)
for i in np.where(tr)[0]: np.add.at(tok_freq, np.asarray(IDS_B[i]), 1)
seen = tok_freq > 0
from scipy.stats import spearmanr
rho = spearmanr(tok_freq[seen], norms[seen]).statistic
print(f"unigram: {UNI_B} rows · {(~seen).mean():.0%} never seen in train · {(norms < 0.01).mean():.0%} effectively dead (‖row‖<0.01)")
print(f"norm–frequency spearman {rho:.2f}   (high = power-law organized already; decay has less to add)")
order = np.argsort(tok_freq[seen]); ns = norms[seen][order]
dec = np.array_split(ns, 10)
print("median ‖row‖ by frequency decile (rare→common):", [round(float(np.median(d)), 2) for d in dec])
print(f"bigram EARNED fraction: {(bnorms > 0.01).mean():.1%} of 16,384 zero-init rows moved — the local context the model bought")

unigram: 32768 rows · 22% never seen in train · 0% effectively dead (‖row‖<0.01)
norm–frequency spearman 0.03   (high = power-law organized already; decay has less to add)
median ‖row‖ by frequency decile (rare→common): [7.86, 7.85, 7.83, 7.81, 7.83, 7.85, 7.88, 7.92, 7.93, 7.89]
bigram EARNED fraction: 100.0% of 16,384 zero-init rows moved — the local context the model bought


In [9]:
# 22 · LABEL PRUNE — co-firing duplicate clusters ("third closing brace in java" ≈ "coding symbols")
A = Y[tr].T.astype(np.float32)                              # latent x train-span firing matrix
inter = A @ A.T; cnt = A.sum(1)
jac = inter / (cnt[:, None] + cnt[None, :] - inter + 1e-9); np.fill_diagonal(jac, 0)
THR = 0.80
parent = list(range(N_V))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a_, b_ in zip(*np.where(np.triu(jac, 1) > THR)): parent[find(int(a_))] = find(int(b_))
groups = {}
for i in range(N_V): groups.setdefault(find(i), []).append(i)
dups = sorted([g for g in groups.values() if len(g) > 1], key=len, reverse=True)
F1S = {}
try: F1S = {json.loads(l)["latent"]: json.loads(l).get("f1", 0) for l in open(f"{BUNDLE}/validated.jsonl")}
except Exception: pass
def qual(c):
    lat = int(V[c]); m = CAPS.get(lat, {})
    return (bool(IS_GATED[c]), F1S.get(lat, 0), m.get("specificity") or 0)
rep = {}
for g in dups:
    best = max(g, key=qual)
    for c in g:
        if c != best: rep[int(V[c])] = int(V[best])
eff = N_V - len(rep)
print(f"co-firing clusters (jaccard>{THR}): {len(dups)} groups · {len(rep)} latents fold away → effective dictionary {eff}/{N_V}")
for g in dups[:10]:
    print("  · " + "  |  ".join(("✓" if IS_GATED[c] else "≈") + CAP_OF[c][:46] for c in g[:3]) + (f"  (+{len(g)-3})" if len(g) > 3 else ""))
json.dump({"threshold": THR, "map": rep}, open(f"{OUTG}/latent_groups.json", "w"))
print("saved latent_groups.json — drop into golf_final/ and the app will collapse duplicates at display")

co-firing clusters (jaccard>0.8): 6 groups · 19 latents fold away → effective dictionary 2029/2048
  · ≈Introductory or heading words for lists, optio  |  ≈Words and prefixes indicating small scale, fra  |  ≈Expressing rarity, simplicity, uniqueness, con  (+5)
  · ✓Focuses on code that sends or writes data, suc  |  ≈Tokens related to energy generation, electrici  |  ≈Renewable energy sources like solar and wind p  (+4)
  · ≈Titles and ranks like Professor, Vice Presiden  |  ≈Tokens related to roles, positions, and offici  |  ≈This latent detects tokens related to problem-  (+1)
  · ✓Detects the preposition 'for' in common phrasa  |  ✓The word 'for' used to indicate purpose or ben
  · ✓Detects the presence of existence, often with   |  ✓Detects the existence of something, often in a
  · ✓Detects Python code that imports modules, espe  |  ✓Detects Python code that imports modules, espe
saved latent_groups.json — drop into golf_final/ and the app will collapse duplicates at display


In [10]:
# 23 · RUNG S — model soup: same init, different data order × 4 = the noise bar AND a free ensemble
# NOTE: set EPOCHS = 40 first if you want members comparable to shipped F (15 = quick look)
MEMBERS, sds, mets = 4, [], []
for m_ in range(MEMBERS):
    torch.manual_seed(0); random.seed(100 + m_); np.random.seed(100 + m_)      # identical init, different shuffles
    rr = train_rung(f"S{m_}", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True)
    sds.append({k: v.cpu().float() for k, v in rr["model"].state_dict().items()}); mets.append(rr["res"]["all"]["z_p5"])
print(f"members z-p@5 {[round(x, 3) for x in mets]} · spread ±{(max(mets) - min(mets)) / 2:.4f}  <- the single-seed noise bar")
avg = {k: torch.stack([sd_[k] for sd_ in sds]).mean(0) for k in sds[0]}
soup = Golf(UNI_B, fdim=FDIM, prior=PRIOR).to(DEV); soup.load_state_dict(avg); soup.eval()
S_s = score_all(soup, IDS_B, TGT_B); res_s, mu_s, sd_s = honest(S_s)
R["SOUP"] = {"model": soup, "S": S_s, "res": res_s, "mu": mu_s, "sd": sd_s, "params": R["F"]["params"]}
print("SOUP: " + " · ".join(f"{k} {v:.3f}" for k, v in res_s["all"].items())
      + " | per-dom " + " ".join(f"{d}:{res_s[d]['z_p5']:.2f}" for d in DOMS if d in res_s))
print("read: soup beats best member on per-domain/OOD -> ship the soup (same 5.3MB, zero inference cost)")

S0: 5.30M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  S0 alive · 20/323 batches · 0.06s/batch -> ~0.3 min/epoch
  S0 ep1 loss 1.4407 · 14s
  S0 ep2 loss 1.0965 · 12s
  S0 ep3 loss 0.9840 · 13s · z-p@5 0.933 (best 0.933)
  S0 ep4 loss 0.9151 · 12s
  S0 ep5 loss 0.8682 · 12s
  S0 ep6 loss 0.8332 · 12s · z-p@5 0.941 (best 0.941)
  S0 ep7 loss 0.8058 · 12s
  S0 ep8 loss 0.7837 · 12s
  S0 ep9 loss 0.7653 · 12s · z-p@5 0.945 (best 0.945)
  S0 ep10 loss 0.7493 · 12s
  S0 ep11 loss 0.7356 · 12s
  S0 ep12 loss 0.7233 · 12s · z-p@5 0.946 (best 0.946)
  S0 ep13 loss 0.7133 · 12s
  S0 ep14 loss 0.7034 · 12s
  S0 ep15 loss 0.6946 · 12s · z-p@5 0.947 (best 0.947)
S0 DONE: 5.30M · z_p5 0.947 · AP_lift 0.469 · macroAUC 0.889 | per-dom z-p@5 web:0.92 code:0.93 cot:0.97 math:0.96 chat:0.98
S1: 5.30M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  S1 alive · 20/323 batches · 0.03s/batch -> ~0.1 min/epoch
  S1 ep1 loss 1.4426 · 12s
  S1 ep2 loss 1.0981 · 12s
  S1 ep3 loss 0.9854 · 12s · 

In [11]:
# 24 · RUNG T — table decay, judged on the SIZE-QUALITY FRONTIER (accuracy alone can't see this win)
import copy
from sklearn.cluster import MiniBatchKMeans
def fq(w, bits):
    if bits >= 16: return w.half().float()
    qmax = 2 ** (bits - 1) - 1
    s_ = w.abs().amax(dim=-1, keepdim=True) / qmax + 1e-12
    return (w / s_).round().clamp(-qmax, qmax) * s_
def frontier(tag):
    model = R[tag]["model"]
    for bits in [8, 4]:
        m2 = copy.deepcopy(model).to(DEV)
        with torch.no_grad():
            for _, p_ in m2.named_parameters(): p_.copy_(fq(p_.data, bits))
        z = honest(score_all(m2, IDS_B, TGT_B))[0]["all"]["z_p5"]
        print(f"  {tag} int{bits}   ~{R[tag]['params'] * bits / 8 / 1e6:5.1f}MB · z-p@5 {z:.3f}")
    m2 = copy.deepcopy(model).cpu(); Wt = m2.uni.weight.data.numpy(); Mp, Kc = 8, 256; sub = Wt.shape[1] // Mp
    Wq = np.zeros_like(Wt)
    for j in range(Mp):
        km = MiniBatchKMeans(Kc, n_init=2, batch_size=8192, max_iter=40, random_state=0).fit(Wt[:, j*sub:(j+1)*sub])
        Wq[:, j*sub:(j+1)*sub] = km.cluster_centers_[km.labels_]
    with torch.no_grad(): m2.uni.weight.copy_(torch.tensor(Wq))
    z = honest(score_all(m2.to(DEV), IDS_B, TGT_B))[0]["all"]["z_p5"]
    mb = (UNI_B * Mp + Mp * Kc * sub * 2 + (R[tag]["params"] - UNI_B * FDIM)) / 1e6
    print(f"  {tag} uni-PQ+int8 ~{mb:5.1f}MB · z-p@5 {z:.3f}")
torch.manual_seed(0); random.seed(0); np.random.seed(0)
R["T"] = train_rung("T", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True, table_wd=1e-2)
print("frontier F (shipped):"); frontier("F")
print("frontier T (table_wd=1e-2):"); frontier("T")
print("read: if T holds z-p@5 at a smaller PQ point, decay bought compressibility — the thing accuracy metrics can't see")

T: 5.30M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  T alive · 20/323 batches · 0.03s/batch -> ~0.1 min/epoch
  T ep1 loss 1.4441 · 12s
  T ep2 loss 1.0986 · 12s
  T ep3 loss 0.9850 · 12s · z-p@5 0.933 (best 0.933)
  T ep4 loss 0.9157 · 12s
  T ep5 loss 0.8675 · 12s
  T ep6 loss 0.8317 · 12s · z-p@5 0.940 (best 0.940)
  T ep7 loss 0.8049 · 12s
  T ep8 loss 0.7819 · 12s
  T ep9 loss 0.7632 · 12s · z-p@5 0.942 (best 0.942)
  T ep10 loss 0.7471 · 12s
  T ep11 loss 0.7334 · 12s
  T ep12 loss 0.7210 · 12s · z-p@5 0.945 (best 0.945)
  T ep13 loss 0.7102 · 12s
  T ep14 loss 0.7007 · 12s
  T ep15 loss 0.6919 · 12s · z-p@5 0.946 (best 0.946)
T DONE: 5.30M · z_p5 0.946 · AP_lift 0.467 · macroAUC 0.888 | per-dom z-p@5 web:0.92 code:0.93 cot:0.97 math:0.95 chat:0.98
frontier F (shipped):
  F int8   ~  5.3MB · z-p@5 0.956
  F int4   ~  2.6MB · z-p@5 0.949
  F uni-PQ+int8 ~  3.5MB · z-p@5 0.870
frontier T (table_wd=1e-2):
  T int8   ~  5.3MB · z-p@5 0.946
  T int4   ~  2.6MB · z-p@5 0.942

In [12]:
# 25 · SURGICAL: zero the never-seen unigram rows in shipped F (no retraining) -> accuracy + compression re-check
import copy
from sklearn.cluster import MiniBatchKMeans
def _fq(w, bits):
    qmax = 2 ** (bits - 1) - 1
    s_ = w.abs().amax(dim=-1, keepdim=True) / qmax + 1e-12
    return (w / s_).round().clamp(-qmax, qmax) * s_
def _frontier(tag):
    model = R[tag]["model"]
    for bits in [8, 4]:
        m2 = copy.deepcopy(model).to(DEV)
        with torch.no_grad():
            for _, p_ in m2.named_parameters(): p_.copy_(_fq(p_.data, bits))
        z = honest(score_all(m2, IDS_B, TGT_B))[0]["all"]["z_p5"]
        print(f"  {tag} int{bits}       ~{R[tag]['params'] * bits / 8 / 1e6:5.1f}MB · z-p@5 {z:.3f}")
    m2 = copy.deepcopy(model).cpu(); Wt = m2.uni.weight.data.numpy(); Mp, Kc = 8, 256; sub = Wt.shape[1] // Mp
    Wq = np.zeros_like(Wt)
    for j in range(Mp):
        km = MiniBatchKMeans(Kc, n_init=2, batch_size=8192, max_iter=40, random_state=0).fit(Wt[:, j*sub:(j+1)*sub])
        Wq[:, j*sub:(j+1)*sub] = km.cluster_centers_[km.labels_]
    with torch.no_grad(): m2.uni.weight.copy_(torch.tensor(Wq))
    z = honest(score_all(m2.to(DEV), IDS_B, TGT_B))[0]["all"]["z_p5"]
    mb = (UNI_B * Mp + Mp * Kc * sub * 2 + (R[tag]["params"] - UNI_B * FDIM)) / 1e6
    print(f"  {tag} uni-PQ+int8 ~{mb:5.1f}MB · z-p@5 {z:.3f}")
tok_freq = np.zeros(UNI_B, np.int64)
for i in np.where(tr)[0]: np.add.at(tok_freq, np.asarray(IDS_B[i]), 1)
mZ = copy.deepcopy(R["F"]["model"]).to(DEV)
with torch.no_grad(): mZ.uni.weight[torch.tensor(tok_freq == 0, device=DEV)] = 0
S_Z = score_all(mZ, IDS_B, TGT_B); resZ, muZ, sdZ = honest(S_Z)
R["FZ"] = {"model": mZ, "S": S_Z, "res": resZ, "mu": muZ, "sd": sdZ, "params": R["F"]["params"]}
print(f"zeroed {(tok_freq == 0).sum()} unseen rows ({(tok_freq == 0).mean():.0%} of table)")
print("FZ: " + " · ".join(f"{k} {v:.3f}" for k, v in resZ["all"].items()) + "   (F was 0.956 / +0.488 / 0.896)")
print("frontier FZ (unseen rows zeroed):"); _frontier("FZ")
print("vs frontier F: int8 0.956 · int4 0.949 · uni-PQ+int8 0.870  <- the PQ point is the one to watch")

zeroed 7358 unseen rows (22% of table)
FZ: z_p5 0.955 · AP_lift 0.488 · macroAUC 0.896   (F was 0.956 / +0.488 / 0.896)
frontier FZ (unseen rows zeroed):
  FZ int8       ~  5.3MB · z-p@5 0.955
  FZ int4       ~  2.6MB · z-p@5 0.949
  FZ uni-PQ+int8 ~  3.5MB · z-p@5 0.873
vs frontier F: int8 0.956 · int4 0.949 · uni-PQ+int8 0.870  <- the PQ point is the one to watch


In [13]:
# 26 · RUNG G — the GOLFED transformer: pre-norm + SwiGLU + RoPE + QK-norm, no dropout, near-zero-init tables
# (equal budget: compare to S-members 0.946 ±0.0005, NOT to 40-epoch F)
class GBlock(nn.Module):
    def __init__(s, d, nh):
        super().__init__(); s.nh, s.hd = nh, d // nh
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.qkv = nn.Linear(d, 3 * d, bias=False); s.out = nn.Linear(d, d, bias=False)
        s.qn, s.kn = nn.LayerNorm(s.hd), nn.LayerNorm(s.hd)              # qk-norm
        h = int(d * 8 / 3 // 16 * 16)                                    # SwiGLU at 2/3 of 4d = param-neutral
        s.w12 = nn.Linear(d, 2 * h, bias=False); s.w3 = nn.Linear(h, d, bias=False)
    def rope(s, x, T):
        half = s.hd // 2
        fr = torch.exp(-math.log(10000.0) * torch.arange(half, device=x.device) / half)
        ang = torch.arange(T, device=x.device)[:, None] * fr[None]
        cos, sin = ang.cos()[None, None], ang.sin()[None, None]
        x1, x2 = x[..., :half], x[..., half:]
        return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1)
    def forward(s, x, m):
        B, T, D = x.shape
        h = s.ln1(x)
        q, k, v = s.qkv(h).view(B, T, 3, s.nh, s.hd).permute(2, 0, 3, 1, 4)
        q, k = s.rope(s.qn(q), T), s.rope(s.kn(k), T)
        att = F.scaled_dot_product_attention(q, k, v, attn_mask=m[:, None, None, :])
        x = x + s.out(att.transpose(1, 2).reshape(B, T, D))
        h = s.ln2(x); a, b = s.w12(h).chunk(2, -1)
        return x + s.w3(F.silu(a) * b)
class GolfG(nn.Module):
    def __init__(s, n_uni, fdim=None, prior=None, smear=False, use_big=True):
        super().__init__(); d = CDIM; e = fdim or d
        s.uni = nn.Embedding(n_uni, e); nn.init.normal_(s.uni.weight, std=0.02)   # NEAR-ZERO init: unseen rows stay tiny
        s.big = nn.Embedding(BIG_H, e); nn.init.zeros_(s.big.weight)
        s.use_big = use_big
        s.proj = nn.Linear(e, d, bias=False)
        s.blocks = nn.ModuleList([GBlock(d, NHEAD) for _ in range(NLAYER)])       # RoPE inside -> NO position table, no 112 cap
        s.lnf = nn.LayerNorm(d)
        s.head = nn.Linear(d, N_V)
        if prior is not None: s.head.bias.data = prior.clone()
    def forward(s, u, b, m):
        x = s.proj(s.uni(u) + (s.big(b) if s.use_big else 0))
        for blk in s.blocks: x = blk(x, m)
        t = s.head(s.lnf(x))
        mk = t.masked_fill(~m.unsqueeze(-1), -1e30); n = m.sum(1, keepdim=True).clamp(min=1).float()
        return torch.logsumexp(mk, 1) - torch.log(n), t
_G0 = Golf; Golf = GolfG
torch.manual_seed(0); random.seed(0); np.random.seed(0)
R["G"] = train_rung("G", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True)
Golf = _G0
u_norms = R["G"]["model"].uni.weight.detach().norm(dim=1).cpu().numpy()
print(f"G unseen-row median norm: {np.median(u_norms[tok_freq == 0]):.3f}  (F had ~7.9 of random noise)")
print(f"verdict at EQUAL budget: G {R['G']['res']['all']['z_p5']:.3f} vs stock-transformer members 0.946 ±0.0005")
print("frontier G:"); _frontier("G")

G: 5.25M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  G alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  G ep1 loss 1.2468 · 13s
  G ep2 loss 0.8471 · 13s
  G ep3 loss 0.7242 · 13s · z-p@5 0.949 (best 0.949)
  G ep4 loss 0.6583 · 13s
  G ep5 loss 0.6147 · 13s
  G ep6 loss 0.5831 · 13s · z-p@5 0.952 (best 0.952)
  G ep7 loss 0.5585 · 13s
  G ep8 loss 0.5381 · 13s
  G ep9 loss 0.5205 · 13s · z-p@5 0.950 (best 0.952)
  G ep10 loss 0.5056 · 13s
  G ep11 loss 0.4931 · 13s
  G ep12 loss 0.4818 · 13s · z-p@5 0.947 (best 0.952)
  G ep13 loss 0.4719 · 13s
  G ep14 loss 0.4623 · 13s
  G ep15 loss 0.4541 · 13s · z-p@5 0.944 (best 0.952)
G DONE: 5.25M · z_p5 0.952 · AP_lift 0.483 · macroAUC 0.893 | per-dom z-p@5 web:0.94 code:0.93 cot:0.97 math:0.96 chat:0.99
G unseen-row median norm: 0.159  (F had ~7.9 of random noise)
verdict at EQUAL budget: G 0.952 vs stock-transformer members 0.946 ±0.0005
frontier G:
  G int8       ~  5.2MB · z-p@5 0.952
  G int4       ~  2.6MB · z-p@5 0.95

In [14]:
# 27 · G-FINAL — golfed block trained to convergence (24 ep cosine; it peaks early) + EXPORT golf2 bundle
EPOCHS_G = 24
torch.manual_seed(0); random.seed(0); np.random.seed(0)
model = GolfG(UNI_B, fdim=FDIM, prior=PRIOR).to(DEV)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-4)
n_batch = int(tr.sum()) // BS + 1
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_G * n_batch)
pw = PW.to(DEV); best = (-1, None)
for ep in range(1, EPOCHS_G + 1):
    model.train(); ls, nb_ = 0.0, 0
    for _, u, b, m, yb, qa in batches(np.where(tr)[0], IDS_B, TGT_B):
        u, b, m, yb, qa = u.to(DEV), b.to(DEV), m.to(DEV), yb.to(DEV), qa.to(DEV)
        _, tok = model(u, b, m)
        loss = F.binary_cross_entropy_with_logits(tok[m], yb[m], pos_weight=pw)
        fire = m & (qa.sum(-1) > 0)
        if fire.any():
            q = qa[fire] / qa[fire].sum(-1, keepdim=True)
            loss = loss + SOFT_W * F.kl_div(F.log_softmax(tok[fire] / TEMP, -1), q, reduction="batchmean")
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        ls += float(loss.detach()); nb_ += 1
    line = f"  GF ep{ep} loss {ls/nb_:.4f}"
    if ep % 2 == 0 or ep == EPOCHS_G:
        S = score_all(model, IDS_B, TGT_B); r, _, _ = honest(S); zp = r["all"]["z_p5"]
        if zp > best[0]: best = (zp, {k: v.cpu().clone() for k, v in model.state_dict().items()})
        line += f" · z-p@5 {zp:.3f} (best {best[0]:.3f})"
    print(line)
model.load_state_dict(best[1]); S = score_all(model, IDS_B, TGT_B)
res, mu, sd = honest(S)
R["GF"] = {"model": model, "S": S, "res": res, "mu": mu, "sd": sd,
           "params": sum(p.numel() for p in model.parameters() if p.requires_grad)}
print("GF DONE: " + " · ".join(f"{k} {v:.3f}" for k, v in res["all"].items())
      + " | per-dom " + " ".join(f"{d}:{res[d]['z_p5']:.2f}" for d in DOMS if d in res))
print("frontier GF:"); _frontier("GF")
# export golf2 (note: cfg carries block="golfG" — the app loader needs the GBlock class to load this)
import pickle, gzip
OUT2 = "/content/golf2"; os.makedirs(OUT2, exist_ok=True)
mdl = R["GF"]["model"].cpu().eval()
mu_dom = {d: R["GF"]["S"][tr & (dom_of == i)].mean(0).tolist() for i, d in enumerate(DOMS)}
sd_dom = {d: (R["GF"]["S"][tr & (dom_of == i)].std(0) + 1e-4).tolist() for i, d in enumerate(DOMS)}
meta = {"rung": "GF", "cfg": {"UNI": UNI_B, "BIG_H": BIG_H, "CDIM": CDIM, "FDIM": FDIM, "NHEAD": NHEAD,
        "NLAYER": NLAYER, "MAXLEN": 4096, "TEMP": TEMP, "block": "golfG"},
        "V": V.tolist(), "captions": CAP_OF, "gated": IS_GATED.tolist(), "prior": PRIOR.tolist(),
        "mu": R["GF"]["mu"].tolist(), "sd": R["GF"]["sd"].tolist(), "mu_dom": mu_dom, "sd_dom": sd_dom,
        "metrics": R["GF"]["res"]["all"]}
torch.save({"state_dict": mdl.state_dict(), **meta}, f"{OUT2}/mega_golf2_probe.pth")
pack = {"meta": meta, "q": {}}
for k, wv in mdl.state_dict().items():
    w = wv.float()
    if w.ndim >= 2:
        s_ = w.abs().amax(-1, keepdim=True) / 127 + 1e-12
        pack["q"][k] = ("i8", (w / s_).round().to(torch.int8).numpy(), s_.squeeze(-1).half().numpy())
    else: pack["q"][k] = ("f16", w.half().numpy(), None)
with gzip.open(f"{OUT2}/mega_golf2_probe_int8.pkl.gz", "wb") as f: pickle.dump(pack, f, 4)
shutil.copy(f"{BUNDLE}/mega_tok.model", OUT2)
if os.path.exists(f"{OUTG}/latent_groups.json"): shutil.copy(f"{OUTG}/latent_groups.json", OUT2)
shutil.make_archive("/content/golf2", "zip", OUT2)
mdl.to(DEV)
print("golf2.zip:", round(os.path.getsize("/content/golf2.zip")/1e6, 2), "MB")
from google.colab import files; files.download("/content/golf2.zip")

  GF ep1 loss 1.2469
  GF ep2 loss 0.8472 · z-p@5 0.947 (best 0.947)
  GF ep3 loss 0.7241
  GF ep4 loss 0.6574 · z-p@5 0.951 (best 0.951)
  GF ep5 loss 0.6125
  GF ep6 loss 0.5793 · z-p@5 0.952 (best 0.952)
  GF ep7 loss 0.5526
  GF ep8 loss 0.5299 · z-p@5 0.950 (best 0.952)
  GF ep9 loss 0.5098
  GF ep10 loss 0.4921 · z-p@5 0.949 (best 0.952)
  GF ep11 loss 0.4755
  GF ep12 loss 0.4606 · z-p@5 0.948 (best 0.952)
  GF ep13 loss 0.4470
  GF ep14 loss 0.4342 · z-p@5 0.946 (best 0.952)
  GF ep15 loss 0.4224
  GF ep16 loss 0.4123 · z-p@5 0.945 (best 0.952)
  GF ep17 loss 0.4028
  GF ep18 loss 0.3942 · z-p@5 0.943 (best 0.952)
  GF ep19 loss 0.3874
  GF ep20 loss 0.3821 · z-p@5 0.944 (best 0.952)
  GF ep21 loss 0.3782
  GF ep22 loss 0.3758 · z-p@5 0.942 (best 0.952)
  GF ep23 loss 0.3744
  GF ep24 loss 0.3736 · z-p@5 0.939 (best 0.952)
GF DONE: z_p5 0.952 · AP_lift 0.483 · macroAUC 0.893 | per-dom web:0.94 code:0.93 cot:0.97 math:0.96 chat:0.99
frontier GF:
  GF int8       ~  5.2MB · z-p@5 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# 28 · RUNG H — a 3rd golfed block (+0.4MB int4) · RUNG I — hungry tables (table LR 3e-3, transformer 1e-3)
# re-run cell 6 first (train_rung gained table_lr); equal budget: compare BOTH to G(2blk) 0.952, noise ±0.0005
_G0 = Golf; Golf = GolfG
_NL = NLAYER; NLAYER = 3
torch.manual_seed(0); random.seed(0); np.random.seed(0)
R["H"] = train_rung("H", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True)
NLAYER = _NL
torch.manual_seed(0); random.seed(0); np.random.seed(0)
R["I"] = train_rung("I", IDS_B, TGT_B, UNI_B, fdim=FDIM, prior=PRIOR, soft=True, table_lr=3e-3)
Golf = _G0
print(f"G 2-blocks 0.952  ·  H 3-blocks {R['H']['res']['all']['z_p5']:.3f} ({R['H']['params']/1e6:.2f}M)"
      f"  ·  I table-lr {R['I']['res']['all']['z_p5']:.3f} ({R['I']['params']/1e6:.2f}M)")
print("read: H-G = is depth worth +0.4MB int4 · I-G = were the tables rate-limited by a shared LR")


H: 6.03M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  H alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  H ep1 loss 1.2389 · 13s
  H ep2 loss 0.8348 · 14s
  H ep3 loss 0.7101 · 14s · z-p@5 0.951 (best 0.951)
  H ep4 loss 0.6426 · 14s
  H ep5 loss 0.5976 · 13s
  H ep6 loss 0.5642 · 14s · z-p@5 0.952 (best 0.952)
  H ep7 loss 0.5380 · 13s
  H ep8 loss 0.5160 · 13s
  H ep9 loss 0.4969 · 14s · z-p@5 0.949 (best 0.952)
  H ep10 loss 0.4812 · 13s
  H ep11 loss 0.4672 · 14s
  H ep12 loss 0.4548 · 14s · z-p@5 0.944 (best 0.952)
  H ep13 loss 0.4437 · 13s
  H ep14 loss 0.4334 · 13s
  H ep15 loss 0.4248 · 13s · z-p@5 0.945 (best 0.952)
H DONE: 6.03M · z_p5 0.952 · AP_lift 0.490 · macroAUC 0.896 | per-dom z-p@5 web:0.93 code:0.94 cot:0.97 math:0.96 chat:0.99


TypeError: train_rung() got an unexpected keyword argument 'table_lr'

In [16]:
# 29 · SHUFFLE NULL — retrain with permuted span→target pairing; the signal MUST collapse to the floors
perm = np.random.default_rng(0).permutation(n_spans)
TGT_NULL = [TGT_B[perm[i]] for i in range(n_spans)]           # every span gets a DIFFERENT span's firings
_G0 = Golf; Golf = GolfG
torch.manual_seed(0); random.seed(0); np.random.seed(0)
R["NULL"] = train_rung("NULL", IDS_B, TGT_NULL, UNI_B, fdim=FDIM, prior=PRIOR, soft=True)
Golf = _G0
try: F5 = FREQ5_FLOOR
except NameError: F5 = float(Y[te][:, np.argsort(-freq)[:5]].mean())
print(f"NULL z-p@5 {R['NULL']['res']['all']['z_p5']:.3f}  vs real G 0.952  vs frequency-5 floor {F5:.3f}  vs random {float(Y[te].mean()):.3f}")
print("read: null ≈ floor proves the 0.95 is text→latent signal, not leakage or base-rate memorization")


NULL: 5.25M trainable · 323 batches/epoch · 15 epochs · lr 0.001
  NULL alive · 20/323 batches · 0.03s/batch -> ~0.2 min/epoch
  NULL ep1 loss 1.8385 · 12s
  NULL ep2 loss 1.7923 · 12s
  NULL ep3 loss 1.7844 · 12s · z-p@5 0.063 (best 0.063)
  NULL ep4 loss 1.7513 · 12s
  NULL ep5 loss 1.7103 · 12s
  NULL ep6 loss 1.6781 · 12s · z-p@5 0.068 (best 0.068)
  NULL ep7 loss 1.6579 · 12s
  NULL ep8 loss 1.6413 · 12s
  NULL ep9 loss 1.6251 · 12s · z-p@5 0.059 (best 0.068)
  NULL ep10 loss 1.6094 · 12s
  NULL ep11 loss 1.5962 · 12s
  NULL ep12 loss 1.5860 · 12s · z-p@5 0.057 (best 0.068)
  NULL ep13 loss 1.5732 · 12s
  NULL ep14 loss 1.5601 · 12s
  NULL ep15 loss 1.5489 · 12s · z-p@5 0.064 (best 0.068)
NULL DONE: 5.25M · z_p5 0.068 · AP_lift -0.001 · macroAUC 0.488 | per-dom z-p@5 web:0.06 code:0.09 cot:0.06 math:0.06 chat:0.06
NULL z-p@5 0.068  vs real G 0.952  vs frequency-5 floor 0.297  vs random 0.067
read: null ≈ floor proves the 0.95 is text→latent signal, not leakage or base-rate memoriz

In [17]:
# 30 · WHICH CONCEPTS FAIL — training dynamics (rare → fixable with data) vs the text-invisible ceiling
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr
S = R["F"]["S"]                                             # the deployed probe
lift = np.full(N_V, np.nan)
for c in range(N_V):
    y = Y[te, c]
    if 0 < y.sum() < y.size:
        lift[c] = average_precision_score(y, S[te, c]) - average_precision_score(y, np.full(y.size, freq[c]))
ok = ~np.isnan(lift); rate = Y[tr].mean(0)
print(f"latents scored {ok.sum()} · median AP-lift {np.nanmedian(lift):+.3f} · at/below floor: {(lift[ok] <= 0).sum()} ({(lift[ok] <= 0).mean():.0%})")
print(f"lift vs frequency spearman {spearmanr(rate[ok], lift[ok]).statistic:+.2f}   (strong positive = rare fails = DATA problem, more harvest fixes it)")
q = np.quantile(rate[ok], np.linspace(0, 1, 11))
print("median lift by frequency decile (rare→common):",
      [round(float(np.nanmedian(lift[ok & (rate >= q[i]) & (rate <= q[i+1])])), 3) for i in range(10)])
print(f"gated median lift {np.nanmedian(lift[ok & IS_GATED]):+.3f} vs ungated {np.nanmedian(lift[ok & ~IS_GATED]):+.3f}")
worst = np.argsort(np.where(ok, lift, 9))[:15]
print()
print("WORST 15 — lift · test-span count · caption · its strongest real firing (is the concept even VISIBLE here?)")
for c in worst:
    m = (tVL == c) & te[tPS]
    ex = spans[int(tPS[m][np.argmax(tPA[m])])]["text"][:84] if m.any() else "(no test firings)"
    print(f"  {lift[c]:+.3f} · n={int(Y[te, c].sum()):4d} · " + ("✓" if IS_GATED[c] else "≈") + CAP_OF[c][:56])
    print(f"           e.g. “{ex}”")
report = [{"latent": int(V[c]), "caption": CAP_OF[c], "gated": bool(IS_GATED[c]), "rate": float(rate[c]),
           "ap_lift": (None if np.isnan(lift[c]) else round(float(lift[c]), 4))} for c in range(N_V)]
json.dump(report, open(f"{OUTG}/concept_report.json", "w"))
print()
print("saved concept_report.json — the per-latent recoverability spectrum, machine-readable")
from google.colab import files; files.download(f"{OUTG}/concept_report.json")


latents scored 2048 · median AP-lift +0.462 · at/below floor: 0 (0%)
lift vs frequency spearman -0.03   (strong positive = rare fails = DATA problem, more harvest fixes it)
median lift by frequency decile (rare→common): [0.466, 0.463, 0.508, 0.433, 0.543, 0.517, 0.5, 0.456, 0.425, 0.403]
gated median lift +0.588 vs ungated +0.390

WORST 15 — lift · test-span count · caption · its strongest real firing (is the concept even VISIBLE here?)
  +0.011 · n= 327 · ≈This latent detects specific tokens related to code, mat
           e.g. “Operators say 'told you so' on iPhone security Operator talking-shop the OMTP (Open ”
  +0.024 · n= 423 · ✓Python code blocks, often starting with 'if __name__ == 
           e.g. “DATABASES = {
 'default': {
 'ENGINE': 'django.db.backends.sqlite3', # Add 'postgres”
  +0.030 · n= 301 · ≈This latent detects tokens related to broadcasting, wire
           e.g. “Broadcast Theme Park Tour program shows various Hallyu contents all in one place.”
  +0.032 · n= 315 ·

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>